In [ ]:
import numpy as np
import oyaml as yaml
import pandas as pd

from prismadv.data_models import Constraints
from prismadv.data_models.validated_results import ValidationResults
from prismadv.project_manager.manager.base import ProjectManager
from prismadv.utils import get_project_root

In [ ]:
# dataset_names = ["playground-series-s4e10", "healthcare_dataset", "hr_analytics", "sleep_health"]
dataset_name_options = ["students", "hr_analytics", "sleep_health", "IPL_win_prediction", "imdb"]
model_order = ['']

In [ ]:
results_df = pd.DataFrame()
for dataset_name in dataset_name_options:
    project_manager = ProjectManager(project_root=get_project_root(), dataset_name=dataset_name)
    subtask_names = project_manager.get_available_subtasks()
    for subtask_name in subtask_names:
        processed_data_labels = project_manager.get_available_processed_data_labels_for_subtask(subtask_name)
        script_path_list = project_manager.get_available_script_path_list_for_subtask(subtask_name)
        script_names = [script_path.stem for script_path in script_path_list]
        for processed_data_label in processed_data_labels:
            if int(processed_data_label) == 0:
                continue
            for script_name in script_names:
                constraint_file = project_manager.get_task_agnostic_constraint_path(
                    subtask_name, processed_data_label) / "stats_novelty_constraints.yaml"
                constraint_validation_results_dir = project_manager.get_task_agnostic_constraints_validation_path(
                    subtask_name, processed_data_label)
                # for constraint_file in constraint_file_list:
                with open(f"{constraint_file}", "r") as f:
                    raw_constraint_dict = yaml.load(f, Loader=yaml.FullLoader)
                constraints = Constraints.from_dict({"constraints": raw_constraint_dict["constraints"]})
                for is_clean in [True, False]:
                    if is_clean:
                        constraint_validation_result_path = constraint_validation_results_dir / f"validation_results_on_clean_test_data__{constraint_file.stem}.yaml"
                    else:
                        constraint_validation_result_path = constraint_validation_results_dir / f"validation_results_on_corrupted_test_data__{constraint_file.stem}.yaml"
                    try:
                        validation_results = ValidationResults.from_yaml(constraint_validation_result_path)
                    except FileNotFoundError:
                        continue
                    num_passed_warning, num_failed_warning, num_failed_error, num_passed_error, num_non_compilable = validation_results.check_result()
                    total_constraints = num_passed_warning + num_failed_warning + num_failed_error + num_passed_error
                    predicted_as_safe = (num_failed_error == 0)

                    execution_result_path = project_manager.get_execution_output_validation_path(
                        subtask_name, processed_data_label, script_name
                    ) / f"basic_metrics_evaluation.json"
                    try:
                        with open(execution_result_path, "r") as f:
                            execution_results = yaml.load(f, Loader=yaml.FullLoader)
                    except FileNotFoundError:
                        continue
                    if is_clean == True:
                        is_safe = execution_results['clean_data_is_safe']
                    else:
                        is_safe = execution_results['corrupted_data_is_safe']
                    new_row = {
                        "llm_name": "",
                        "llm_temperature": np.nan,
                        "dataset_name": dataset_name,
                        "subtask_name": subtask_name,
                        "processed_data_label": processed_data_label,
                        "script_name": script_name,
                        "is_clean": is_clean,
                        "num_passed_warning": num_passed_warning,
                        "num_failed_warning": num_failed_warning,
                        "num_failed_error": num_failed_error,
                        "num_passed_error": num_passed_error,
                        "num_non_compilable": num_non_compilable,
                        "total_constraints": total_constraints,
                        "predicted_as_safe": predicted_as_safe,
                        "is_safe": is_safe
                    }
                    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
df_all_unique = results_df

In [ ]:
from workflow_prismadv.utils.results_analysis import format_results_latex, build_confusion_matrices

confusion_matrices_df = build_confusion_matrices(df_all_unique)

In [ ]:
latex_output = format_results_latex(
    confusion_matrex_df=confusion_matrices_df,
    dataset_name_options=dataset_name_options,
    model_order=model_order,
    decimals=1,
    model_label_prefix="stats-novelty",
    include_overall=True
)
print(latex_output)

In [ ]:
from workflow_prismadv.utils.results_analysis import build_correct_grids, plot_correct_grids

grids = build_correct_grids(df_all_unique, dataset_name_options, threshold=0.5)
plot_correct_grids(grids, model_order=model_order, dataset_name_options=dataset_name_options)